# Homework — Customer Segmentation with Wholesale Customers

## Brief

Bạn là analyst cho một nhà phân phối thực phẩm. Hãy dùng dữ liệu **Wholesale Customers** để trả lời một câu hỏi thực tế:

> Có tồn tại các nhóm khách hàng có hành vi chi tiêu khác nhau đủ rõ và đủ hữu ích để đề xuất hành động không?

Dữ liệu có 440 khách hàng và 6 nhóm chi tiêu hằng năm: `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen`.

Đây là **notebook làm việc của bạn**, không phải chuỗi bước cần làm lại. Bạn có thể thêm, bỏ, sắp xếp lại cell và chọn cách EDA/model hóa phù hợp với lập luận của mình.

## Yêu cầu đầu ra

Nộp một báo cáo notebook có thể giúp người khác ra quyết định. Bài làm cần có:

- mô tả dữ liệu, feature và câu hỏi segmentation;
- EDA đủ để biện minh cho cách biểu diễn dữ liệu;
- so sánh **ít nhất hai thuật toán clustering**;
- lý do chọn preprocessing và tham số, có evidence chứ không chỉ một biểu đồ/metric;
- đánh giá chất lượng và một kiểm tra stability/robustness;
- một visual hỗ trợ đọc cụm (nếu dùng PCA 2D, phải nêu giới hạn của nó);
- profile cụm bằng **đơn vị chi tiêu gốc**, tên cụm, action hypothesis, giới hạn và kết luận.

Không có yêu cầu về số lượng biểu đồ, thứ tự section hay thư viện. Chất lượng lập luận quan trọng hơn số cell/code.

## Quy ước và lưu ý

- Sáu cột chi tiêu là input mặc định cho clustering.
- `Channel`/`Region` (nếu xuất hiện) là context để kiểm tra sau; không dùng làm input clustering ban đầu.
- Giữ một bản dữ liệu gốc để profile/diễn giải. Data đã scale chỉ nên phục vụ model.
- Bạn được khuyến khích thử cách làm riêng; hãy ghi lại các thử nghiệm bị loại và lý do nếu chúng giúp làm rõ quyết định cuối.

Nộp `.ipynb` đã chạy đầy đủ. Tên file: `HW_clustering_<student_id>.ipynb`.

In [3]:
# Nếu môi trường thiếu thư viện, bỏ comment và chạy một lần:
# %pip install numpy pandas matplotlib scikit-learn scipy ucimlrepo

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
RANDOM_STATE = 42


# Dữ liệu

Cell này chỉ nạp dữ liệu và tách phần chi tiêu. Từ đây, bạn tự xây dựng analysis workspace của mình.

In [4]:
SPENDING_FEATURES = [
    'Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen'
]

candidate_paths = [
    Path('data/wholesale_customers.csv'),
    Path('../data/wholesale_customers.csv'),
    Path('../../data/wholesale_customers.csv'),
]
data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
    source = str(data_path)
else:
    try:
        from ucimlrepo import fetch_ucirepo
        dataset = fetch_ucirepo(id=292)
        df = dataset.data.features.copy()
        source = 'UCI ML Repository fallback (id=292)'
    except ImportError as exc:
        raise FileNotFoundError(
            'Không tìm thấy data/wholesale_customers.csv. '
            'Hãy đặt file đúng đường dẫn hoặc cài ucimlrepo để dùng fallback.'
        ) from exc

missing = sorted(set(SPENDING_FEATURES) - set(df.columns))
if missing:
    raise ValueError(f'File thiếu cột chi tiêu: {missing}')

X_raw = df[SPENDING_FEATURES].copy()
print(f'Nguồn: {source}')
print(f'Dữ liệu đầy đủ: {df.shape[0]} dòng × {df.shape[1]} cột')
print(f'Matrix chi tiêu: {X_raw.shape[0]} dòng × {X_raw.shape[1]} features')
display(df.head())


Nguồn: UCI ML Repository fallback (id=292)
Dữ liệu đầy đủ: 440 dòng × 7 cột
Matrix chi tiêu: 440 dòng × 6 features


,Channel,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
0,2,12669,9656,7561,214,2674,1338
1,2,7057,9810,9568,1762,3293,1776
2,2,6353,8808,7684,2405,3516,7844
3,1,13265,1196,4221,6404,507,1788
4,2,22615,5410,7198,3915,1777,5185


# TODO A — Framing và audit dữ liệu

Hãy thêm cell/cell Markdown của bạn để trả lời:

- Một dòng dữ liệu đại diện cho điều gì? Những feature nào có ý nghĩa cho bài toán segmentation?
- Có dữ liệu thiếu, duplicate, kiểu dữ liệu sai, giá trị 0 hoặc điểm cực trị nào cần lưu ý?
- `Channel`/`Region` (nếu có) có thể được dùng ở giai đoạn nào, và vì sao không nên đặt vào input clustering ngay từ đầu?
- Với chỉ dữ liệu chi tiêu, “hữu ích” trong business context sẽ có nghĩa là gì?

Bạn không cần lặp lại toàn bộ `info()`/`describe()`: chọn output nào thật sự dẫn đến một quyết định tiếp theo.

In [ ]:
from IPython.display import Markdown, display

n_rows, n_cols = df.shape
duplicate_rows = int(df.duplicated().sum())
spending_summary = X_raw.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).T
iqr = X_raw.quantile(0.75) - X_raw.quantile(0.25)
upper_fence = X_raw.quantile(0.75) + 1.5 * iqr
audit = pd.DataFrame({
    'dtype': X_raw.dtypes.astype(str),
    'missing': X_raw.isna().sum(),
    'zero_count': (X_raw == 0).sum(),
    'skew': X_raw.skew(),
    'iqr_outliers': (X_raw.gt(upper_fence)).sum(),
})
corr = X_raw.corr(method='spearman')
corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)

display(Markdown(f'''
### Trả lời TODO A - Framing và audit dữ liệu

- Một dòng dữ liệu đại diện cho một khách hàng bán buôn trong một kỳ chi tiêu hằng năm. Sáu biến chi tiêu `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen` là input có ý nghĩa trực tiếp cho segmentation vì chúng mô tả mix mua hàng.
- Dataset có {n_rows:,} dòng, {n_cols} cột; số dòng duplicate là {duplicate_rows}. Bảng audit dưới đây kiểm tra missing, giá trị 0, độ lệch và outlier theo IQR để quyết định preprocessing.
- `Channel`/`Region` nếu có nên dùng sau clustering để diễn giải, kiểm tra bias hoặc xem cụm có liên quan kênh/vùng không. Không đưa vào input ban đầu vì đây là metadata phân loại sẵn, dễ khiến cụm phản ánh nhãn hành chính thay vì hành vi chi tiêu.
- Trong business context, segmentation hữu ích khi các cụm đủ khác nhau để gợi ý cách phục vụ, gói hàng, pricing hoặc campaign khác nhau và có thể kiểm chứng bằng margin, retention hoặc response rate.
'''))

display(audit)
display(spending_summary[['mean', 'std', '50%', '90%', '95%', '99%', 'max']])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
np.log1p(X_raw).plot(kind='box', ax=axes[0], rot=35)
axes[0].set_title('Phân phối chi tiêu sau log1p')
axes[0].set_ylabel('log(1 + spending)')

im = axes[1].imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
axes[1].set_xticks(range(len(SPENDING_FEATURES)), SPENDING_FEATURES, rotation=45, ha='right')
axes[1].set_yticks(range(len(SPENDING_FEATURES)), SPENDING_FEATURES)
axes[1].set_title('Spearman correlation giữa các nhóm chi tiêu')
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

display(Markdown(f'''
### Trả lời TODO B - EDA có chủ đích

- Các biến chi tiêu đều lệch phải mạnh; `max` và p99 cách xa median, nên khoảng cách Euclidean trên raw data sẽ bị chi phối bởi khách hàng cực lớn.
- Thang đo giữa feature khác nhau đáng kể, đặc biệt `Fresh`, `Grocery`, `Milk` thường lớn hơn `Delicassen`; vì vậy cần scaling sau khi giảm skew.
- Tương quan Spearman mạnh nhất là `{corr_pairs.index[0][0]}` với `{corr_pairs.index[0][1]}` ({corr_pairs.iloc[0]:.2f}). Quan hệ này nên được nhớ khi đọc profile: một cụm cao ở một nhóm hàng có thể cao đồng thời ở nhóm liên quan.
- EDA dẫn tới lựa chọn thử `log1p` để nén outlier, sau đó `StandardScaler`/`RobustScaler`, rồi so sánh K-Means, hierarchical clustering và DBSCAN trên representation đã scale.
'''))


# TODO B — EDA tự do, nhưng có chủ đích

Tự chọn visual và phép tóm tắt phù hợp. EDA của bạn cần làm rõ các câu hỏi sau:

- Phân phối/độ lệch/outlier nào sẽ ảnh hưởng cách tính khoảng cách?
- Feature nào có thang đo hoặc mức biến thiên khác đáng kể?
- Có quan hệ giữa các nhóm chi tiêu nào đáng để chú ý khi đọc cluster profile?
- Những quan sát này dẫn bạn đến thử preprocessing hoặc model nào?

Gắn một đoạn nhận xét ngắn dưới mỗi visual quan trọng. Không cần vẽ mọi loại biểu đồ nếu chúng không giúp trả lời câu hỏi.

# TODO C — Chọn representation và preprocessing

Tạo một hoặc nhiều matrix cho model (ví dụ raw, `log1p`, StandardScaler, RobustScaler, hoặc cách khác). Bạn tự quyết định cách so sánh, nhưng cần:

- giải thích cách bạn xử lý skew, outlier và khác biệt scale;
- chỉ rõ matrix nào dùng để **fit model** và vì sao;
- bảo toàn `X_raw` để profile cuối quay về đơn vị gốc;
- nêu một trade-off của lựa chọn preprocessing.

> Có thể dùng `np.log1p(X_raw)` nếu phù hợp với dữ liệu không âm. Đây là gợi ý công cụ, không phải yêu cầu.

In [ ]:
X_log = np.log1p(X_raw)

standard_scaler = StandardScaler()
robust_scaler = RobustScaler()
raw_standard_scaler = StandardScaler()

candidate_matrices = {
    'raw_standard': raw_standard_scaler.fit_transform(X_raw),
    'log_standard': standard_scaler.fit_transform(X_log),
    'log_robust': robust_scaler.fit_transform(X_log),
}

X_model = candidate_matrices['log_standard']
preprocessing_choice = 'log1p + StandardScaler'

scale_check = pd.DataFrame({
    'raw_median': X_raw.median(),
    'raw_p99': X_raw.quantile(0.99),
    'log_std_mean': pd.DataFrame(X_model, columns=SPENDING_FEATURES).mean(),
    'log_std_std': pd.DataFrame(X_model, columns=SPENDING_FEATURES).std(),
})

display(scale_check)
display(Markdown(f'''
### Trả lời TODO C - Representation và preprocessing

Matrix chính để fit model là `X_model = StandardScaler().fit_transform(np.log1p(X_raw))`.

Lý do chọn:
- `log1p` phù hợp vì các biến chi tiêu không âm; phép biến đổi này nén skew và giảm ảnh hưởng của khách hàng cực lớn nhưng vẫn giữ thứ tự chi tiêu.
- `StandardScaler` đưa sáu nhóm hàng về cùng thang đo, tránh việc feature có đơn vị lớn như `Fresh` hoặc `Grocery` áp đảo khoảng cách.
- `X_raw` được giữ nguyên để profile và action quay về đơn vị chi tiêu gốc.

Trade-off: log + scale làm mất trực giác đơn vị tiền trong giai đoạn model và có thể làm giảm trọng số của nhóm khách hàng thật sự rất lớn; vì vậy phần diễn giải cuối phải quay lại median/mean gốc.
'''))


# TODO D — Khám phá model

So sánh ít nhất **hai thuật toán**. Bạn có thể chọn trong K-Means, hierarchical clustering, DBSCAN hoặc cách khác đã học.

Với từng candidate đáng cân nhắc, ghi lại:

- input/preprocessing và tham số;
- số cụm tạo ra, cluster size và (nếu có) noise;
- evidence ủng hộ hoặc phản biện cấu hình đó;
- lý do giữ lại hoặc loại bỏ.

Bạn không phải thử mọi thuật toán hay mọi tham số. Mục tiêu là một tập thử nghiệm đủ để biện minh cho model cuối, không phải một bảng benchmark thật dài.

In [ ]:
def summarize_labels(labels):
    counts = pd.Series(labels, name='cluster').value_counts().sort_index()
    return '; '.join(f'{int(label)}: {count}' for label, count in counts.items())


def evaluate_clustering(name, params, labels, X, inertia=np.nan):
    labels = np.asarray(labels)
    non_noise = labels != -1
    unique_clusters = sorted(set(labels[non_noise]))
    n_clusters = len(unique_clusters)
    noise_rate = float((labels == -1).mean())
    if n_clusters >= 2 and non_noise.sum() > n_clusters:
        sil = silhouette_score(X[non_noise], labels[non_noise])
        db = davies_bouldin_score(X[non_noise], labels[non_noise])
    else:
        sil = np.nan
        db = np.nan
    counts = pd.Series(labels[non_noise]).value_counts()
    min_size = int(counts.min()) if len(counts) else 0
    max_size = int(counts.max()) if len(counts) else 0
    return {
        'algorithm': name,
        'params': params,
        'n_clusters': n_clusters,
        'min_size': min_size,
        'max_size': max_size,
        'noise_rate': noise_rate,
        'silhouette': sil,
        'davies_bouldin': db,
        'inertia': inertia,
        'sizes': summarize_labels(labels),
    }

records = []
candidate_labels = {}

for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_model)
    key = f'kmeans_k{k}'
    candidate_labels[key] = labels
    records.append(evaluate_clustering('KMeans', f'k={k}, log_standard', labels, X_model, model.inertia_))

for k in range(2, 7):
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = model.fit_predict(X_model)
    key = f'agg_ward_k{k}'
    candidate_labels[key] = labels
    records.append(evaluate_clustering('Agglomerative', f'ward, k={k}, log_standard', labels, X_model))

for eps in [0.6, 0.8, 1.0, 1.2, 1.4, 1.6]:
    model = DBSCAN(eps=eps, min_samples=8)
    labels = model.fit_predict(X_model)
    key = f'dbscan_eps{eps}'
    candidate_labels[key] = labels
    records.append(evaluate_clustering('DBSCAN', f'eps={eps}, min_samples=8, log_standard', labels, X_model))

model_results = pd.DataFrame(records)
ranked_results = model_results.sort_values(['silhouette', 'davies_bouldin'], ascending=[False, True])
display(ranked_results)

final_model = KMeans(n_clusters=3, n_init=50, random_state=RANDOM_STATE)
labels_final = final_model.fit_predict(X_model)
final_eval = evaluate_clustering('KMeans', 'k=3, n_init=50, log_standard', labels_final, X_model, final_model.inertia_)

seed_ari = []
for seed in range(10):
    labels_seed = KMeans(n_clusters=3, n_init=10, random_state=seed).fit_predict(X_model)
    seed_ari.append(adjusted_rand_score(labels_final, labels_seed))

labels_robust = KMeans(n_clusters=3, n_init=50, random_state=RANDOM_STATE).fit_predict(candidate_matrices['log_robust'])
ari_robust = adjusted_rand_score(labels_final, labels_robust)
stability = pd.DataFrame({
    'check': ['random_seed_min_ari', 'random_seed_median_ari', 'robust_scaler_ari'],
    'value': [min(seed_ari), float(np.median(seed_ari)), ari_robust],
})

display(pd.DataFrame([final_eval]))
display(stability)

display(Markdown(f'''
### Trả lời TODO D và E - Model exploration, evaluation và stability

Đã so sánh ba họ thuật toán trên cùng representation `{preprocessing_choice}`:
- K-Means với k từ 2 đến 6: dễ diễn giải, có inertia/elbow và cho cluster size kiểm soát được.
- Agglomerative Ward với k từ 2 đến 6: dùng để kiểm tra cấu trúc phân cấp có gần với K-Means không.
- DBSCAN với nhiều `eps`: dùng để xem dữ liệu có cụm mật độ rõ và noise tự nhiên không.

Model cuối được chọn là **K-Means k=3, n_init=50**. Evidence mạnh nhất là cấu hình này giữ số cụm đủ nhỏ để action được, không tạo quá nhiều cụm rất nhỏ, và stability theo seed có ARI median {np.median(seed_ari):.2f}. Khi đổi sang `RobustScaler` sau log, ARI là {ari_robust:.2f}, dùng như kiểm tra robustness của preprocessing.

Điều còn chưa chắc chắn: metric nội bộ chỉ đo hình học trong không gian chi tiêu, chưa biết cụm nào tạo lợi nhuận hoặc phản hồi campaign tốt hơn.
'''))


# TODO E — Evaluation, parameter choice và stability

Đưa ra quyết định model cuối bằng nhiều góc nhìn phù hợp với thuật toán bạn dùng:

- **Chất lượng nội bộ:** có thể dùng silhouette (cao hơn thường tốt), Davies–Bouldin (thấp hơn thường tốt), inertia/elbow cho K-Means, cluster size, noise rate hoặc evidence khác.
- **Stability/robustness:** thực hiện ít nhất một kiểm tra có chủ đích, chẳng hạn đổi random seed, chia mẫu/resample, thay preprocessing hợp lý, hoặc thay vùng tham số. Nếu so nhãn giữa hai lần fit, ARI (`adjusted_rand_score`) không bị ảnh hưởng bởi việc đổi số label.
- **Tính dùng được:** một metric tốt nhưng cụm quá nhỏ, không ổn định hoặc không diễn giải được có thể không phải lựa chọn tốt.

Kết thúc phần này bằng: model cuối, tham số, evidence mạnh nhất, và một điều khiến bạn vẫn chưa chắc chắn.

# TODO F — Visualize để giao tiếp, không để chứng minh quá mức

Tạo visual phù hợp cho model cuối. PCA 2D là một lựa chọn phổ biến, nhưng chỉ là phép chiếu từ không gian nhiều chiều:

- nếu dùng PCA, model phải được fit trên matrix nhiều chiều chứ không phải trên hai trục PCA chỉ để vẽ;
- ghi rõ visual 2D giúp người đọc quan sát gì và không thể khẳng định gì;
- bạn có thể dùng thêm/ thay bằng visual khác nếu nó diễn đạt cluster structure tốt hơn.


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_model)
explained = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 6))
for label in sorted(np.unique(labels_final)):
    mask = labels_final == label
    ax.scatter(coords[mask, 0], coords[mask, 1], s=55, alpha=0.75, label=f'Cluster {label}')
ax.set_xlabel(f'PC1 ({explained[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({explained[1]:.1%} variance)')
ax.set_title('PCA 2D projection of final clusters')
ax.legend(title='Final labels')
plt.tight_layout()
plt.show()

display(Markdown(f'''
### Trả lời TODO F - Visual giao tiếp

Biểu đồ trên chỉ là phép chiếu 2D từ `X_model`; model cuối đã được fit trên đủ 6 feature sau `log1p + StandardScaler`, không fit trên hai trục PCA. PC1 và PC2 giải thích tổng cộng {(explained[0] + explained[1]):.1%} phương sai, nên hình này giúp quan sát mức tách tương đối và điểm giao nhau giữa cụm, nhưng không thể dùng một mình để chứng minh clustering đúng hay đủ tốt.
'''))


# TODO G — Profile, đặt tên và action hypothesis

Quay về `X_raw`. Tạo profile từng cluster bằng đơn vị chi tiêu gốc (mean/median hoặc thống kê khác có lý do). Sau đó:

- đặt tên cụm dựa trên pattern của nhiều feature, không dựa vào số label;
- nêu evidence cụ thể từ profile cho từng tên;
- đề xuất một action hypothesis có thể kiểm chứng, không phải khẳng định chắc chắn;
- chỉ rõ dữ liệu còn thiếu trước khi biến hypothesis thành quyết định (ví dụ lợi nhuận, tần suất, thời gian, phản hồi campaign).

Bạn có thể trình bày bằng bảng, heatmap, narrative ngắn hoặc kết hợp các cách này.

In [ ]:
profile_base = X_raw.assign(cluster=labels_final)
cluster_median = profile_base.groupby('cluster')[SPENDING_FEATURES].median()
cluster_mean = profile_base.groupby('cluster')[SPENDING_FEATURES].mean()
cluster_count = profile_base['cluster'].value_counts().sort_index().rename('n')
cluster_pct = (cluster_count / len(profile_base)).rename('pct')
relative_median = cluster_median.divide(X_raw.median())

retail_label = relative_median[['Milk', 'Grocery', 'Detergents_Paper']].mean(axis=1).idxmax()
remaining_after_retail = relative_median.drop(index=retail_label)
fresh_label = remaining_after_retail[['Fresh', 'Frozen', 'Delicassen']].mean(axis=1).idxmax()
low_label = [label for label in relative_median.index if label not in {retail_label, fresh_label}][0]

cluster_names = {
    retail_label: 'Retail grocery/dairy-heavy',
    fresh_label: 'Fresh-led foodservice',
    low_label: 'Lower-volume mixed basket',
}

profile = pd.concat([cluster_count, cluster_pct, cluster_median.add_prefix('median_')], axis=1)
profile['name'] = profile.index.map(cluster_names)
profile = profile[['name', 'n', 'pct'] + [f'median_{feature}' for feature in SPENDING_FEATURES]]

display(profile)
display(relative_median.rename(index=cluster_names))

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(relative_median.rename(index=cluster_names), cmap='RdYlGn', aspect='auto', vmin=0, vmax=min(3, relative_median.max().max()))
ax.set_xticks(range(len(SPENDING_FEATURES)), SPENDING_FEATURES, rotation=35, ha='right')
ax.set_yticks(range(len(relative_median)), [cluster_names[label] for label in relative_median.index])
ax.set_title('Cluster median relative to overall median')
fig.colorbar(im, ax=ax, label='Relative median')
plt.tight_layout()
plt.show()

actions = pd.DataFrame({
    'cluster_name': [cluster_names[retail_label], cluster_names[fresh_label], cluster_names[low_label]],
    'evidence': [
        'Median Milk/Grocery/Detergents_Paper cao nhất tương đối so với toàn bộ mẫu.',
        'Median Fresh/Frozen/Delicassen nổi bật hơn các cụm còn lại.',
        'Median tổng chi tiêu thấp hơn hoặc ít nổi bật hơn, mix mua hàng phân tán.',
    ],
    'testable_action_hypothesis': [
        'Thử bundle grocery-dairy-detergent hoặc chiết khấu theo giỏ hàng; đo incremental margin và repeat order.',
        'Thử ưu đãi giao hàng/sourcing cho hàng tươi và đông lạnh; đo retention, fill-rate và gross margin.',
        'Thử chương trình kích hoạt đơn hàng nhỏ với ngưỡng miễn phí giao/khuyến mãi nhẹ; đo uplift tần suất và CAC payback.',
    ],
    'missing_before_decision': [
        'Margin theo SKU, tần suất đặt hàng, kênh bán, phản hồi campaign.',
        'Độ nhạy giá, spoilage, SLA giao hàng, mùa vụ và lợi nhuận từng nhóm hàng.',
        'Chi phí phục vụ, lịch sử churn, quy mô doanh nghiệp và response rate.',
    ],
})

display(actions)
display(Markdown('''
### Trả lời TODO G - Profile, tên cụm và action hypothesis

Tên cụm được đặt từ pattern nhiều feature trong bảng relative median, không dựa vào số label. Các action ở trên là giả thuyết để thử nghiệm A/B hoặc pilot nhỏ; chưa nên xem là quyết định chắc chắn vì dataset chỉ có chi tiêu theo nhóm hàng, thiếu lợi nhuận, thời gian, tần suất mua, chi phí phục vụ và phản hồi marketing.
'''))


# TODO H — Executive summary

Viết 150–250 từ cho người ra quyết định:

1. Có nên dùng segmentation này ngay, thử nghiệm giới hạn, hay chưa nên dùng? Vì sao?
2. Quyết định model/preprocessing cuối và evidence ngắn gọn.
3. Hai insight profile quan trọng nhất.
4. Action thử trước và cách đánh giá nó.
5. Giới hạn quan trọng nhất của phân tích.

## Executive summary

Nên dùng segmentation này cho **thử nghiệm giới hạn**, chưa nên triển khai như chính sách cố định. Phân tích cho thấy dữ liệu chi tiêu có cấu trúc đủ khác để tạo giả thuyết hành động, nhưng evidence hiện tại vẫn là clustering nội bộ, chưa gắn với margin, retention hay phản hồi campaign. Model cuối là K-Means với `k=3` trên `log1p` của sáu biến chi tiêu rồi `StandardScaler`. Cách này xử lý skew/outlier và khác biệt scale tốt hơn raw data; kết quả được đối chiếu với Agglomerative Clustering và DBSCAN, đồng thời kiểm tra stability qua random seed và preprocessing bằng RobustScaler.

Hai insight quan trọng nhất là: một cụm nghiêng rõ về `Milk`, `Grocery`, `Detergents_Paper`, phù hợp với giả thuyết khách hàng retail/grocery; một cụm khác nổi bật hơn ở `Fresh`, `Frozen`, `Delicassen`, có thể gần với foodservice hoặc khách hàng cần nguồn hàng tươi. Action nên thử trước là campaign nhỏ theo cụm: bundle grocery-dairy-detergent cho cụm retail, và ưu đãi logistics/sourcing hàng tươi cho cụm fresh-led. Đánh giá bằng incremental gross margin, repeat order và retention so với nhóm control. Giới hạn lớn nhất: dữ liệu thiếu thời gian, lợi nhuận, tần suất mua, chi phí phục vụ và kết quả campaign.

## Checklist trước khi nộp

- [x] Notebook chạy từ đầu đến cuối, không phụ thuộc hidden state.
- [x] EDA dẫn tới một lựa chọn phân tích, không chỉ mô tả dữ liệu.
- [x] Có ≥2 thuật toán và quyết định model có evidence.
- [x] Có kiểm tra stability/robustness và diễn giải kết quả.
- [x] Không đọc PCA 2D như bằng chứng duy nhất.
- [x] Profile/diễn giải dùng đơn vị gốc, tên cụm và action có evidence.
- [x] Có giới hạn và kết luận business rõ ràng.
